# supervised variant prediction for multiple genes

In [1]:
# ======================  STANDARD LIB / TORCH / SKLEARN  =====================
import os, ast, json, functools, warnings
from pathlib import Path
import numpy as np, pandas as pd
from tqdm import tqdm
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
# -----------------------------------------------------------------------------

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV,RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import roc_auc_score
from scipy.stats import randint, uniform
import torch.optim as optim

In [2]:
# ------------------------------- PATHS ---------------------------------------
BASE        = Path(".")
EMB_DIR     = ".data/embeddings"
SEQ_DIR     = ".data/sequence_data_csv"
OUT_DIR     = "results"

# # ----------------------------- GENE‑SET MAP ----------------------------------




In [3]:
# ------------------------------ LOADERS --------------------------------------

import glob, numpy as np, pandas as pd, os

def load_esm(gene, phenotype_csv_override=None, embedding_dir_override=None):
    """
    Load ESM embeddings for a given gene and match with phenotype labels.

    Parameters:
    - gene: str, gene name (used for default paths)
    - phenotype_csv_override: optional CSV path to override default phenotype file
    - embedding_dir_override: optional name override if embeddings are saved under a different folder (e.g., 'inhA_ETH')

    Returns:
    - DataFrame with columns: identifier, emb_{gene}, Phenotype_{gene}
    """
    gene_for_embedding = embedding_dir_override if embedding_dir_override else gene
    chunk_glob = f"data/embeddings/{gene_for_embedding}/mean/{gene_for_embedding}_mean_chunk_*.npz"
    chunk_paths = sorted(glob.glob(chunk_glob))
    if not chunk_paths:
        raise FileNotFoundError(f"No .npz found for: {chunk_glob}")

    embeds, ids_all = [], []
    for p in chunk_paths:
        z = np.load(p)
        embeds.append(z["mean_embedding"])
        ids_all.extend(z["identifier"].astype(str))

    X = np.vstack(embeds)

    # Match phenotype labels
    if phenotype_csv_override is not None:
        csv_path = phenotype_csv_override
    else:
        csv_path = f"data/sequence_data_csv/{gene}_combined_sequence_data.csv"
    pheno_df = pd.read_csv(csv_path, usecols=["Filename", "Phenotype"]).set_index("Filename")
    y = pheno_df.loc[ids_all, "Phenotype"].values

    return pd.DataFrame({
        "identifier": ids_all,
        f"emb_{gene}": list(X)
    }).assign(**{f"Phenotype_{gene}": y})



def _parse(v):
    return np.asarray(v if isinstance(v,(list,np.ndarray)) else ast.literal_eval(v),
                      dtype=np.float32)

# def load_seq(g):
#     f = f"data/sequence_data_csv/{g}_combined_sequence_data.csv"
#     df = pd.read_csv(f, usecols=["Filename","Phenotype","Protein_Sequence"])
#     return df.rename(columns={
#         "Filename"        :"identifier",
#         "Phenotype"       :f"Phenotype_{g}",
#         "Protein_Sequence":f"seq_{g}",
#     })
def load_seq(g):
    f = f"data/sequence_data_csv/{g}_combined_sequence_data.csv"
    df = pd.read_csv(f, usecols=["Filename", "Phenotype", "Protein_Sequence", "Frameshift_Mutation"])
    return df.rename(columns={
        "Filename"           : "identifier",
        "Phenotype"          : f"Phenotype_{g}",
        "Protein_Sequence"   : f"seq_{g}",
        "Frameshift_Mutation": f"frameshift_{g}"
    })

# def merge_genes(genes):
#     frames=[]
#     for g in genes:
#         frames.append(pd.merge(load_esm(g), load_seq(g), on="identifier"))
#     df = functools.reduce(lambda l,r: pd.merge(l,r,on="identifier",how="inner"), frames)
#     ph_cols = [c for c in df.columns if c.startswith("Phenotype_")]
#     df["Phenotype_final"] = df[ph_cols].mode(axis=1)[0]
#     df = df[df[ph_cols].nunique(axis=1)==1]   # drop conflicts
#     return df.reset_index(drop=True)

def merge_genes(genes, embedding_gene_override=None, phenotype_csv_override=None):
    """
    Args:
        genes: list of gene names (e.g., ['ethA', 'inhA'])
        embedding_gene_override: dict mapping gene → override folder name for embeddings
        phenotype_csv_override: dict mapping gene → custom CSV path with phenotypes
    """
    frames = []
    for g in genes:
        emb_g = embedding_gene_override.get(g, g) if embedding_gene_override else g
        pheno_path = phenotype_csv_override.get(g, None) if phenotype_csv_override else None

        df = pd.merge(load_esm(emb_g, phenotype_csv_override=pheno_path), load_seq(g), on="identifier")
        frames.append(df)

    df = functools.reduce(lambda l, r: pd.merge(l, r, on="identifier", how="inner"), frames)

    ph_cols = [c for c in df.columns if c.startswith("Phenotype_")]
    df["Phenotype_final"] = df[ph_cols].mode(axis=1)[0]
    df = df[df[ph_cols].nunique(axis=1) == 1]
    
    emb_cols_used = [f"emb_{embedding_gene_override.get(g, g) if embedding_gene_override else g}" for g in genes]
    seq_cols_used = [f"seq_{g}" for g in genes]
    # return df.reset_index(drop=True)
    return df.reset_index(drop=True), emb_cols_used, seq_cols_used


In [4]:
#  Step 5: PCA Visualization and Save
def visualize_pca(X_train, y_train, gene_name, num_components=60):
    """
    Perform PCA and save the visualization.
    """
    pca = PCA(n_components=num_components)
    X_train_pca = pca.fit_transform(X_train)

    plt.figure(figsize=(7, 6))
    sc = plt.scatter(X_train_pca[:, 0], X_train_pca[:, 1], c=y_train, marker='.')
    plt.xlabel("PCA First Principal Component")
    plt.ylabel("PCA Second Principal Component")
    plt.colorbar(sc, label="Variant Effect")
    plt.title(f"PCA Visualization for {gene_name}")

    # Save PCA Figure
    pca_path = os.path.join(RESULTS_DIR, f"{gene_name}_pca.png")
    plt.savefig(pca_path)
    plt.close()
    
    return pca


# ------------------------------------------------------------
#  RandomizedSearchCV instead of GridSearchCV  (ASCII only!)
# ------------------------------------------------------------

def run_random_search(
        X_train, y_train,
        num_pca_components: int = 60,
        n_iter: int = 25,            # total random combos per model
        cv_folds: int = 3,
        random_state: int = 42
):
    """
    Faster hyper‑parameter tuning with RandomizedSearchCV.
    """

    # ---------- search spaces (all ASCII hyphens) -----------------
    knn_dist = {
        'model': [KNeighborsRegressor()],
        'model__n_neighbors': randint(3, 31),            # 3–30
        'model__weights': ['uniform', 'distance'],
        'model__algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
        'model__leaf_size': randint(10, 50),              # 10–49
        'model__p': randint(1, 3)                         # 1 or 2
    }

    svr_dist = {
            'model': [SVR()],
            'model__C': [0.1, 1.0],
            'model__kernel': ['linear', 'rbf'],
            'model__degree': [3],
            'model__gamma': ['scale'],
        }
    rf_dist = {
        'model': [RandomForestRegressor()],
        'model__n_estimators': randint(50, 301),          # 50–300
        'model__max_features': ['sqrt', 'log2'],
        'model__min_samples_split': randint(2, 11),       # 2–10
        'model__min_samples_leaf': randint(1, 6),         # 1–5
        'model__criterion': ['squared_error']
    }

    search_spaces = [knn_dist, svr_dist, rf_dist]
    models        = [KNeighborsRegressor, SVR, RandomForestRegressor]

    pipe = Pipeline([('pca', PCA(num_pca_components)),
                     ('model', 'passthrough')])

    results, searches = [], []

    for cls, param_dist in zip(models, search_spaces):
        print(f"Running Randomized Search for {cls.__name__} …")

        rs = RandomizedSearchCV(
            estimator           = pipe,
            param_distributions = param_dist,
            n_iter              = n_iter,
            cv                  = cv_folds,
            random_state        = random_state,
            n_jobs              = -1,
            verbose             = 1,
            scoring             = 'r2'
        )
        rs.fit(X_train, y_train)
        results.append(pd.DataFrame(rs.cv_results_))
        searches.append(rs)

    return results, searches







#  Step 8: Evaluate Best Models using AUC and Save Results
def evaluate_models(grid_list, X_test, y_test, gene_name):
    """
    Evaluate the best models using AUC and save results.
    """
    results = []

    for grid in grid_list:
        best_model = grid.best_estimator_.get_params()["steps"][1][1]
        preds = grid.predict(X_test)

        try:
            auc = roc_auc_score(y_test, preds)
        except ValueError as e:
            print(f"[Warning] AUC could not be computed for {gene_name}: {e}")
            auc = None

        results.append({
            "Gene": gene_name,
            "Best Model": str(best_model),
            "AUC": auc
        })

        print(f"Best Model for {gene_name}: {best_model}")
        if auc is not None:
            print(f"AUC: {auc:.4f}")
        else:
            print("AUC: N/A (only one class present in test set)")
        print("\n", "-" * 80, "\n")

    return results



In [5]:
# -------------------------- CLASSICAL ML HELPERS -----------------------------
CLASSICAL = {
    "knn": (KNeighborsClassifier(), {"model__n_neighbors":randint(3, 31)} ),
    "svm": (SVC(probability=True),  {"model__C":[0.1,10],"model__kernel":["rbf","linear"]}),
    "rf" : (RandomForestClassifier(),{"model__n_estimators":[200],
                                      "model__max_features":["sqrt"]} )
}
def train_classical(X,y):
    Xtr,Xte,ytr,yte = train_test_split(X,y,stratify=y,test_size=0.2,random_state=42)
    rows=[]
    for name,(base,grid) in CLASSICAL.items():
        pipe = Pipeline([("pca",PCA(n_components=min(100,Xtr.shape[1]))),
                         ("model",base)])
        rs = RandomizedSearchCV(pipe, grid, n_iter=8, cv=5, scoring="roc_auc",
                                n_jobs=-1, random_state=0)
        rs.fit(Xtr,ytr)
        y_hat = rs.best_estimator_.predict_proba(Xte)[:,1]
        
        rows.append({"model":name,
                     "cv_auc": rs.best_score_,
                     "test_auc": roc_auc_score(yte,y_hat)})
    return rows



In [6]:
# ----------------------------- CNN & TRANSFORMER -----------------------------

AA_TO_INDEX = {aa: idx for idx, aa in enumerate("ACDEFGHIKLMNPQRSTVWY")}
class ProteinDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

        lengths = [len(seq) for seq in sequences]
        min_len = min(lengths)
        max_len = max(lengths)

        if max_len - min_len > 1:
            raise ValueError(f"Sequences vary too much in length! Found lengths: {set(lengths)}")

        self.seq_len = max_len  # allow minor difference (pad shorter sequences)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]

        onehot = np.zeros((20, self.seq_len), dtype=np.float32)
        for i, aa in enumerate(seq):
            if i >= self.seq_len:  # (safety) but should never happen
                break
            if aa in AA_TO_INDEX:
                onehot[AA_TO_INDEX[aa], i] = 1.0

        return torch.tensor(onehot), torch.tensor(label, dtype=torch.float32)


# ------------------------- Transformer Model -------------------------
class ProteinTransformer(nn.Module):
    def __init__(self, input_dim=20, d_model=128, nhead=4, num_layers=2):
        super(ProteinTransformer, self).__init__()
        self.embedding = nn.Linear(input_dim, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=256, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(d_model, 1),
            nn.Sigmoid()
        )

    def _generate_positional_encoding(self, d_model, length, device):
        pe = torch.zeros(length, d_model, device=device)
        position = torch.arange(0, length, dtype=torch.float, device=device).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, device=device).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)  # [1, length, d_model]

    def forward(self, x):
        x = x.permute(0, 2, 1)  # [B, L, C]
        x = self.embedding(x)   # [B, L, d_model]

        pos_enc = self._generate_positional_encoding(x.size(2), x.size(1), x.device)  # [1, L, d_model]
        x = x + pos_enc  # broadcast over batch

        x = self.transformer_encoder(x)  # [B, L, d_model]
        x = x.permute(0, 2, 1)  # [B, d_model, L]
        return self.classifier(x).squeeze()


from sklearn.metrics import roc_auc_score

def train_dl(model, dl, epochs=6, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr)
    lossfn = nn.BCELoss()
    model.cuda().train()

    for _ in range(epochs):
        for X, y in dl:
            X, y = X.cuda(), y.cuda()
            opt.zero_grad()
            out = model(X)
            loss = lossfn(out, y)
            loss.backward()
            opt.step()

    # Evaluate AUC on a held-out batch (simplified check)
    model.eval()
    with torch.no_grad():
        X, y = next(iter(dl))
        X, y = X.cuda(), y.cuda()
        preds = model(X)
        auc = roc_auc_score(y.cpu().numpy(), preds.cpu().numpy())

    return auc

class ProteinCNN(nn.Module):
    def __init__(self, input_dim=20):
        super(ProteinCNN, self).__init__()
        self.conv1 = nn.Conv1d(input_dim, 64, kernel_size=5, padding=2)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool(x).squeeze(-1)
        x = self.sigmoid(self.fc(x))
        return x.squeeze()


In [7]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from torch.cuda.amp import autocast

# -------------------------
# LOO Attribution -R + Subsampled S for LOO Attribution
# -------------------------
@torch.no_grad()
def compute_patchwise_loo_attribution_fp16(model, input_tensor, device="cuda", patch_size=5):
    model.eval()
    input_tensor = input_tensor.to(device)
    input_tensor = input_tensor.clone()

    original_output = model(input_tensor).item()
    _, _, L = input_tensor.shape
    deltas = np.zeros(L)

    for start in range(0, L, patch_size):
        end = min(start + patch_size, L)
        masked_input = input_tensor.clone()
        masked_input[0, :, start:end] = 0.0

        with autocast():
            masked_output = model(masked_input).float().item()

        delta = original_output - masked_output
        deltas[start:end] = delta / (end - start)

    return deltas


In [8]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm


# -------------------------
# LOO Attribution
# -------------------------

@torch.no_grad()
def compute_batch_loo_attribution(model, input_tensor, device="cuda"):
    """
    Compute LOO attribution by masking each residue in parallel (batch-efficient).
    
    Args:
        model: Trained CNN or Transformer model.
        input_tensor: torch.Tensor of shape [1, 20, L] (one-hot encoded sequence).
        device: CUDA or CPU device.

    Returns:
        np.array of shape (L,) with LOO importance scores (Δ logits).
    """
    model.eval()
    input_tensor = input_tensor.to(device)  # [1, 20, L]
    input_tensor = input_tensor.clone()

    B, C, L = input_tensor.shape  # B=1, C=20

    # Compute original prediction
    original_output = model(input_tensor).item()

    # Prepare batch of L inputs each with one position masked
    batch_inputs = input_tensor.repeat(L, 1, 1)  # [L, 20, L]
    for i in range(L):
        batch_inputs[i, :, i] = 0.0  # zero out i-th residue

    # Run forward pass
    batch_outputs = model(batch_inputs.to(device))  # shape: [L]
    if batch_outputs.ndim == 0:
        batch_outputs = batch_outputs.unsqueeze(0)
    loo_scores = original_output - batch_outputs.detach().cpu().numpy()

    return loo_scores  # shape: (L,)

In [ ]:
# ------------------------------- MAIN LOOP -----------------------------------
esm_rows, raw_rows = [], []

GENE_SETS = {
    "ethA+ethR"   : ["ethA", "ethR"],
    # "embABC"      : ["embA", "embB", "embC"],
    # "gyrAB"       : ["gyrA", "gyrB"],
    "katG+inhA"   : ["katG", "inhA"]
}

for gs_name,genes in GENE_SETS.items():
    print("gene names", genes)
    df = merge_genes(genes)
    # Construct filename: e.g., embA_embB_embC_merged.csv
    gene_str = "_".join(genes)
    save_path = f"merged_data/{gene_str}_merged.csv"
    
    # Ensure the directory exists
    os.makedirs("merged_data", exist_ok=True)
    
    # Save to CSV
    df.to_csv(save_path, index=False)
    print(f" Saved merged dataframe to {save_path} — {len(df)} rows")
    # Compute embedding column names specific to this gene set
    emb_cols = [f"emb_{g}" for g in genes]
    print("Available columns:", df.columns.tolist())
    print("Expected embedding columns:", emb_cols)

    
    # Step 1: parse all embeddings
    for col in emb_cols:
        df[col] = df[col].apply(_parse)

    # Step 2: stack combined embedding
    X_emb = np.stack([np.concatenate([row[col] for col in emb_cols]) for _, row in df.iterrows()])

    # Step 3: get binary labels
    y = (df["Phenotype_final"] == "R").astype(int).values
    # ---- branch A : ESM + classical ----------------------------------------

    # Step 4: train classical ML
    for r in train_classical(X_emb, y):
        esm_rows.append(dict(gene_set=gs_name, **r))

    # ---- branch B : raw sequence + DL --------------------------------------
    seq_cols=[f"seq_{g}" for g in genes]
    concat_seq = df[seq_cols].agg("".join, axis=1).tolist()
    max_L = max(len(s) for s in concat_seq)
    ds = ProtDS(concat_seq, y, max_L)
    dl = DataLoader(ds,batch_size=64,shuffle=True)

    cnn_auc  = train_dl(ProteinCNN(),    dl)
    tr_auc = train_dl(ProteinTransformer(input_dim=20, max_len=max_L), dl)
    raw_rows.append({"gene_set":gs_name,"model":"cnn","test_auc":cnn_auc})
    raw_rows.append({"gene_set":gs_name,"model":"transformer","test_auc":tr_auc})

# ----------------------------- SAVE RESULTS ----------------------------------
pd.DataFrame(esm_rows).to_csv(OUT_DIR / "esm_classical_auc.csv",   index=False)
pd.DataFrame(raw_rows).to_csv(OUT_DIR / "rawseq_auc.csv",          index=False)
print(" done → results/esm_classical_auc.csv & results/rawseq_auc.csv")


In [ ]:
ethA_df = load_esm("ethA")
inhA_df = load_esm(
    "inhA", 
    phenotype_csv_override="data/sequence_data_csv/inhA_ETH_combined_sequence_data.csv",
    embedding_dir_override="inhA_ETH"
)
# Then you can merge as before
merged_df = pd.merge(ethA_df, inhA_df, on="identifier")
print(f"Merged shape: {merged_df.shape}")

In [ ]:
gs_name = "ethA_inhA"
esm_rows, raw_rows = [], []
genes=['ethA', 'inhA_ETH']
df = merge_genes(
    genes=["ethA", "inhA"],
    embedding_gene_override={"inhA": "inhA_ETH"},
    phenotype_csv_override={"inhA": "data/sequence_data_csv/inhA_ETH_combined_sequence_data.csv"}
)

In [ ]:
df.columns

In [ ]:
# Compute embedding column names specific to this gene set
emb_cols = [f"emb_{g}" for g in genes]
print("Available columns:", df.columns.tolist())
print("Expected embedding columns:", emb_cols)


# Step 1: parse all embeddings
for col in emb_cols:
    df[col] = df[col].apply(_parse)

# Step 2: stack combined embedding
X_emb = np.stack([np.concatenate([row[col] for col in emb_cols]) for _, row in df.iterrows()])

# Step 3: get binary labels
y = (df["Phenotype_final"] == "R").astype(int).values
# ---- branch A : ESM + classical ----------------------------------------

# Step 4: train classical ML
for r in train_classical(X_emb, y):
    esm_rows.append(dict(gene_set=gs_name, **r))

In [ ]:
# ---- branch B : raw sequence + DL --------------------------------------
genes=['ethA', 'inhA']
seq_cols=[f"seq_{g}" for g in genes]
concat_seq = df[seq_cols].agg("".join, axis=1).tolist()
max_L = max(len(s) for s in concat_seq)
ds = ProtDS(concat_seq, y, max_L)
dl = DataLoader(ds,batch_size=64,shuffle=True)

cnn_auc  = train_dl(ProteinCNN(),    dl)
tr_auc = train_dl(ProteinTransformer(input_dim=20, max_len=max_L), dl)
raw_rows.append({"gene_set":gs_name,"model":"cnn","test_auc":cnn_auc})
raw_rows.append({"gene_set":gs_name,"model":"transformer","test_auc":tr_auc})

In [ ]:
# ----------------------------- SAVE RESULTS ----------------------------------
pd.DataFrame(esm_rows).to_csv(OUT_DIR / "ethA_inhA_esm_classical_auc.csv",   index=False)
pd.DataFrame(raw_rows).to_csv(OUT_DIR / "ethA_inhA_rawseq_auc.csv",          index=False)
print("done → results/esm_classical_auc.csv & results/rawseq_auc.csv")


## significance testing

In [14]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import os
import joblib

def crossval_model_auc(X_all, y_all, model_names, run_random_search, gene_name, out_dir):
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    all_auc_records = []
    model_folds = {name: [] for name in model_names}
    
    for fold_id, (train_idx, test_idx) in enumerate(kf.split(X_all, y_all)):
        print(f"\n--- Fold {fold_id + 1} ---")
        X_train, X_test = X_all[train_idx], X_all[test_idx]
        y_train, y_test = y_all[train_idx], y_all[test_idx]

        _, search_list = run_random_search(X_train, y_train, n_iter=25, cv_folds=3)
        
        for model_name, grid in zip(model_names, search_list):
            best_model = grid.best_estimator_
            model_folds[model_name].append(best_model)

            try:
                preds = best_model.predict(X_test)
                auc = roc_auc_score(y_test, preds)
            except ValueError as e:
                auc = None
                print(f"[Warning] AUC error for {model_name}: {e}")

            all_auc_records.append({
                "Gene": gene_name,
                "Model": model_name,
                "Fold": fold_id + 1,
                "AUC": auc
            })

            print(f"Model: {model_name} | Fold {fold_id+1} AUC: {auc}")

            # Save model
            os.makedirs(out_dir, exist_ok=True)
            model_out = os.path.join(out_dir, f"{gene_name}_fold{fold_id+1}_{model_name}.joblib")
            joblib.dump(best_model, model_out)
    
    results_df = pd.DataFrame(all_auc_records)
    results_df.to_csv(os.path.join(out_dir, f"{gene_name}_cv_auc_per_fold.csv"), index=False)
    return results_df


In [15]:
def train_eval_transformer_kfold_multigene(gene_set_name, df, seq_cols, n_splits=5):
    df = df[df["Frameshift_Mutation"] == 0]
    df = df[df["Phenotype_final"].isin(["R", "S"])]

    sequences = df[seq_cols].agg("".join, axis=1).tolist()
    labels = (df["Phenotype_final"] == "R").astype(int).tolist()
    max_len = max(len(seq) for seq in sequences)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    results = []
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    for fold, (train_idx, test_idx) in enumerate(skf.split(sequences, labels), 1):
        print(f"[{gene_set_name} | Transformer] Fold {fold}")
        X_train = [sequences[i] for i in train_idx]
        y_train = [labels[i] for i in train_idx]
        X_test = [sequences[i] for i in test_idx]
        y_test = [labels[i] for i in test_idx]

        model = ProteinTransformer()
        model.to(device)

        train_loader = DataLoader(ProteinDataset(X_train, np.array(y_train)), batch_size=32, shuffle=True)
        criterion = nn.BCELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

        model.train()
        for epoch in range(10):
            for X, y in train_loader:
                X, y = X.to(device), y.to(device)
                optimizer.zero_grad()
                out = model(X)
                loss = criterion(out, y)
                loss.backward()
                optimizer.step()

        # Evaluate
        model.eval()
        test_loader = DataLoader(ProteinDataset(X_test, np.array(y_test)), batch_size=32)
        preds, truths = [], []
        with torch.no_grad():
            for X, y in test_loader:
                out = model(X.to(device)).cpu()
                preds.extend(out.numpy())
                truths.extend(y.numpy())
        auc = roc_auc_score(truths, preds)

        # Save model and fold outputs
        fold_dir = f"transformer_results/{gene_set_name}/fold_{fold}"
        os.makedirs(fold_dir, exist_ok=True)
        torch.save(model.state_dict(), f"{fold_dir}/model.pt")

        # Compute LOO attribution
        loo_all = []
        for X, _ in DataLoader(ProteinDataset(X_test, np.array(y_test)), batch_size=1):
            score = compute_batch_loo_attribution(model, X.to(device))  # Same as CNN version
            loo_all.append(score)
        loo_array = np.array(loo_all)
        np.save(f"{fold_dir}/loo.npy", loo_array)

        # Mean LOO per residue
        mean_loo = loo_array.mean(axis=0)
        pd.DataFrame({
            "Residue_Position": list(range(len(mean_loo))),
            "LOO_Importance": mean_loo
        }).to_csv(f"{fold_dir}/loo_importance.csv", index=False)

        # Save AUC
        pd.DataFrame({"Fold": [fold], "AUC": [auc]}).to_csv(f"{fold_dir}/auc.csv", index=False)
        results.append({"Gene": gene_set_name, "Fold": fold, "AUC": auc})

    # Final summary
    all_results_df = pd.DataFrame(results)
    all_results_df.to_csv(f"transformer_results/{gene_set_name}_fold_auc_summary.csv", index=False)
    return all_results_df


In [16]:
def train_eval_cnn_kfold_multigene(gene_set_name, df, seq_cols, n_splits=5):
    df = df[df["Frameshift_Mutation"] == 0]
    df = df[df["Phenotype_final"].isin(["R", "S"])]
    
    # Concatenate sequences from all genes
    sequences = df[seq_cols].agg("".join, axis=1).tolist()
    labels = (df["Phenotype_final"] == "R").astype(int).tolist()
    max_len = max(len(seq) for seq in sequences)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    results = []
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    for fold, (train_idx, test_idx) in enumerate(skf.split(sequences, labels), 1):
        print(f"[{gene_set_name}] Fold {fold}")
        X_train = [sequences[i] for i in train_idx]
        y_train = [labels[i] for i in train_idx]
        X_test = [sequences[i] for i in test_idx]
        y_test = [labels[i] for i in test_idx]

        # Train model
        model = ProteinCNN()
        model.to(device)
        criterion = nn.BCELoss()
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        train_loader = DataLoader(ProteinDataset(X_train, y_train), batch_size=32, shuffle=True)

        model.train()
        for epoch in range(10):
            for X, y in train_loader:
                X, y = X.to(device), y.to(device)
                optimizer.zero_grad()
                out = model(X)
                loss = criterion(out, y)
                loss.backward()
                optimizer.step()

        # Evaluate
        model.eval()
        test_loader = DataLoader(ProteinDataset(X_test, y_test), batch_size=32)
        preds, truths = [], []
        with torch.no_grad():
            for X, y in test_loader:
                out = model(X.to(device)).cpu()
                preds.extend(out.numpy())
                truths.extend(y.numpy())
        auc = roc_auc_score(truths, preds)

        # Save fold outputs
        fold_dir = f"cnn_results/{gene_set_name}/fold_{fold}"
        os.makedirs(fold_dir, exist_ok=True)
        torch.save(model.state_dict(), f"{fold_dir}/model.pt")

        # Compute LOO for each test sequence
        loo_all = []
        for X, _ in DataLoader(ProteinDataset(X_test, y_test), batch_size=1):
            score = compute_batch_loo_attribution(model, X.to(device))
            loo_all.append(score)
        loo_array = np.array(loo_all)
        np.save(f"{fold_dir}/loo.npy", loo_array)

        # Save mean LOO attribution
        mean_loo = loo_array.mean(axis=0)
        pd.DataFrame({
            "Residue_Position": list(range(len(mean_loo))),
            "LOO_Importance": mean_loo
        }).to_csv(f"{fold_dir}/loo_importance.csv", index=False)

        # Save AUC
        pd.DataFrame({"Fold": [fold], "AUC": [auc]}).to_csv(f"{fold_dir}/auc.csv", index=False)
        results.append({"Gene": gene_set_name, "Fold": fold, "AUC": auc})

    # Save all fold-level AUCs
    all_results_df = pd.DataFrame(results)
    all_results_df.to_csv(f"cnn_results/{gene_set_name}_fold_auc_summary.csv", index=False)
    return all_results_df


In [19]:
GENE_SETS = {
    # "ethA+ethR"   : ["ethA", "ethR"],
    "embABC"      : ["embA", "embB", "embC"],
    # "rpsL+gid": ['rpsL','gid'],
    # "gyrAB"       : ["gyrA", "gyrB"],
    # "katG+inhA"   : ["katG", "inhA"]
}
esm_rows, raw_rows = [], []
cnn_results_all, transformer_results_all= [],[]


In [ ]:
for gs_name, genes in GENE_SETS.items():
    print("gene names", genes)

    df, emb_cols, seq_cols = merge_genes(genes)
    # Parse embeddings
    for col in emb_cols:
        df[col] = df[col].apply(_parse)
        
        
    # In merge_genes() or right after
    frameshift_cols = [f"frameshift_{g}" for g in genes if f"frameshift_{g}" in df.columns]
    if frameshift_cols:
        df["Frameshift_Mutation"] = df[frameshift_cols].max(axis=1)  # conservative: drop if any gene has frameshift
    else:
        df["Frameshift_Mutation"] = 0  # assume no frameshift if not loaded


    emb_cols = [f"emb_{g}" for g in genes]
    for col in emb_cols:
        df[col] = df[col].apply(_parse)
    X_emb = np.stack([np.concatenate([row[col] for col in emb_cols]) for _, row in df.iterrows()])
    y = (df["Phenotype_final"] == "R").astype(int).values


    # # ---- A: Cross-validation with classical ML models ----
    # model_names = ["knn", "svr", "rf"]
    # result_df = crossval_model_auc(
    #     X_all=X_emb,
    #     y_all=y,
    #     model_names=model_names,
    #     run_random_search=run_random_search,
    #     gene_name=gs_name,
    #     out_dir="trained_models"
    # )

    # CNN
    # cnn_result_df = train_eval_cnn_kfold_multigene(gs_name, df, seq_cols)
    # cnn_results_all.append(cnn_result_df)

    # Transformer
    tr_result_df = train_eval_transformer_kfold_multigene(gs_name, df, seq_cols)
    transformer_results_all.append(tr_result_df)

    # Collect for global summary
    # esm_rows.extend(result_df.to_dict(orient="records"))

#     # ---- B: DL on raw sequences ----
#     seq_cols=[f"seq_{g}" for g in genes]
#     concat_seq = df[seq_cols].agg("".join, axis=1).tolist()
#     max_L = max(len(s) for s in concat_seq)
#     ds = ProtDS(concat_seq, y, max_L)
#     dl = DataLoader(ds,batch_size=64,shuffle=True)

#     cnn_auc  = train_dl(ProteinCNN(),    dl)
#     tr_auc = train_dl(ProteinTransformer(input_dim=20, max_len=max_L), dl)
#     raw_rows.append({"gene_set":gs_name,"model":"cnn","test_auc":cnn_auc})
#     raw_rows.append({"gene_set":gs_name,"model":"transformer","test_auc":tr_auc})


gene names ['embA', 'embB', 'embC']
[embABC | Transformer] Fold 1
[embABC | Transformer] Fold 2
